# 面试问题：Agent 调用敏感工具时，怎样用 JIT Credential Broker 避免把长期密钥交给模型？

**一句话回答。** 模型只能提出结构化动作，不能读取真实 API key、refresh token 或云凭据。确定性策略层先校验用户身份、租户、工具、参数、风险和审批，再签发绑定 audience、action digest、scope、TTL 与 nonce 的一次性 capability；credential broker 在隔离执行器内部把 capability 换成短期服务凭据并直接调用工具，密钥永不进入 prompt、tool schema、trace 或模型输出。

本 Notebook 只用虚构标识符实现授权状态机，不生成、保存或使用任何真实凭据。断言用于验证最小权限、时效、参数绑定和重放拒绝，不代表生产密码学或企业 IAM 已经实现。

**资料入口。** [澳大利亚网络安全中心的 Agentic AI 采用指南](https://www.cyber.gov.au/business-government/secure-design/artificial-intelligence/careful-adoption-of-agentic-ai-services) 明确建议为高影响动作使用 just-in-time credentials；MCP 授权规范也强调 audience 验证和禁止 token passthrough。

In [ ]:
question = "Agent JIT credential broker 与无密钥工具执行"  # 执行本行的状态、计算或校验逻辑。
assert "credential" in question  # 执行本行的状态、计算或校验逻辑。
assert 15 // 5 == 3  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 把模型、策略层、凭据代理和执行器分开

模型负责从用户意图产生候选 action；policy decision point 判断是否允许；credential broker 保管身份关系并签发短期能力；tool executor 在受控网络中完成调用。四者应使用不同身份与日志边界。若模型能够直接读取 vault secret，任何 prompt injection、工具输出污染或调试日志都可能升级为凭据泄露。

In [ ]:
action = {"run_id": "run-7", "user": "u-7", "tenant": "t-1", "tool": "billing.refund", "args": {"order_id": "o-9", "amount": 80}, "risk": "high"}  # 执行本行的状态、计算或校验逻辑。
assert action["tool"] == "billing.refund"  # 执行本行的状态、计算或校验逻辑。
assert action["args"]["amount"] == 80  # 执行本行的状态、计算或校验逻辑。
assert "secret" not in action  # 执行本行的状态、计算或校验逻辑。

## 2. 授权基于真实主体与精确参数，而非 Agent 自称

runtime 从已认证会话取得 user/tenant，不接受模型在参数中伪造身份。策略同时检查工具 scope、订单归属、金额上限、环境和审批；高风险退款的批准必须绑定 order、amount 与 policy revision。只批准“允许退款”过于宽泛，模型改动金额后可能继续复用旧授权。

In [ ]:
policy = {"revision": "p-4", "tenant": "t-1", "allowed_tools": {"billing.refund"}, "max_amount": 100, "requires_approval": {"billing.refund"}}  # 执行本行的状态、计算或校验逻辑。
def policy_allows(action_value, policy_value, approved):  # 执行本行的状态、计算或校验逻辑。
    return action_value["tenant"] == policy_value["tenant"] and action_value["tool"] in policy_value["allowed_tools"] and action_value["args"]["amount"] <= policy_value["max_amount"] and (action_value["tool"] not in policy_value["requires_approval"] or approved)  # 执行本行的状态、计算或校验逻辑。
assert policy_allows(action, policy, True)  # 执行本行的状态、计算或校验逻辑。
assert not policy_allows({**action, "tenant": "t-2"}, policy, True)  # 执行本行的状态、计算或校验逻辑。
assert not policy_allows({**action, "args": {**action["args"], "amount": 120}}, policy, True)  # 执行本行的状态、计算或校验逻辑。

## 3. Action Digest 防止批准后参数替换

capability 不能只写 tool name，还要覆盖规范化参数、主体、租户、运行 id 和策略版本。生产会使用标准序列化与密码学摘要；教学用确定性字符串展示同一参数得到同一 digest、金额或订单变化得到不同 digest。执行器只接受与当前 action 完全匹配的票据。

In [ ]:
def action_digest(action_value, policy_revision):  # 执行本行的状态、计算或校验逻辑。
    args = action_value["args"]  # 执行本行的状态、计算或校验逻辑。
    return "|".join((action_value["run_id"], action_value["user"], action_value["tenant"], action_value["tool"], args["order_id"], str(args["amount"]), policy_revision))  # 执行本行的状态、计算或校验逻辑。
digest = action_digest(action, policy["revision"])  # 执行本行的状态、计算或校验逻辑。
assert digest == action_digest(action, "p-4")  # 执行本行的状态、计算或校验逻辑。
assert digest != action_digest({**action, "args": {**action["args"], "amount": 81}}, "p-4")  # 执行本行的状态、计算或校验逻辑。
assert digest.endswith("p-4")  # 执行本行的状态、计算或校验逻辑。

## 4. Capability 是短期、一次性且 audience-bound

策略通过后签发的不是上游长期密钥，而是 broker 自己理解的一次性授权票据。claims 至少包含 subject、tenant、audience、scope、action digest、policy revision、issued/expiry、nonce 和审批 id。TTL 应覆盖一次调用而不是整个长任务；不同资源必须取得不同 capability。

In [ ]:
capability = {"sub": "u-7", "tenant": "t-1", "aud": "billing.example", "scope": {"refund:create"}, "action_digest": digest, "policy": "p-4", "iat": 10, "exp": 20, "nonce": "n-1", "approval": "approval-8"}  # 执行本行的状态、计算或校验逻辑。
assert capability["aud"] == "billing.example"  # 执行本行的状态、计算或校验逻辑。
assert capability["exp"] - capability["iat"] == 10  # 执行本行的状态、计算或校验逻辑。
assert capability["action_digest"] == digest  # 执行本行的状态、计算或校验逻辑。

## 5. Broker 在执行前验证 claims，并且拒绝重放

broker 不相信 Agent 说票据有效，而是独立检查 audience、scope、主体、租户、精确动作摘要、过期时间、策略版本与 nonce 使用状态。nonce 在第一次成功兑换时原子标记 consumed；超时后的不确定结果要先查业务状态，不能重新签发同动作并盲目重试。

In [ ]:
consumed = set()  # 执行本行的状态、计算或校验逻辑。
def redeem(ticket, action_value, audience, scope, now, consumed_value):  # 执行本行的状态、计算或校验逻辑。
    valid = ticket["aud"] == audience and scope in ticket["scope"] and ticket["action_digest"] == action_digest(action_value, ticket["policy"]) and ticket["tenant"] == action_value["tenant"] and ticket["exp"] > now and ticket["nonce"] not in consumed_value  # 执行本行的状态、计算或校验逻辑。
    if valid: consumed_value.add(ticket["nonce"])  # 执行本行的状态、计算或校验逻辑。
    return valid  # 执行本行的状态、计算或校验逻辑。
assert redeem(capability, action, "billing.example", "refund:create", 12, consumed)  # 执行本行的状态、计算或校验逻辑。
assert not redeem(capability, action, "billing.example", "refund:create", 13, consumed)  # 执行本行的状态、计算或校验逻辑。
assert "n-1" in consumed  # 执行本行的状态、计算或校验逻辑。

## 6. Secretless 表示执行器代调用，而不是把 secret 返回给 Agent

broker 可在隔离进程中从 vault 取得短期服务身份，然后直接向目标 API 发请求；模型只看到结构化成功、失败或可公开的业务结果。不能把 access token 当作 tool result 返回，也不能将其拼入 URL、异常、shell 命令或追踪。下游服务需要独立 audience token，禁止透传 Agent 侧 token。

In [ ]:
vault = {"billing-service": "opaque-secret-never-shown"}  # 执行本行的状态、计算或校验逻辑。
def broker_execute(action_value, vault_value):  # 执行本行的状态、计算或校验逻辑。
    secret_used = "billing-service" in vault_value  # 执行本行的状态、计算或校验逻辑。
    return {"status": "accepted", "order_id": action_value["args"]["order_id"], "amount": action_value["args"]["amount"], "credential_exposed": False, "secret_used": secret_used}  # 执行本行的状态、计算或校验逻辑。
result = broker_execute(action, vault)  # 执行本行的状态、计算或校验逻辑。
assert result["status"] == "accepted"  # 执行本行的状态、计算或校验逻辑。
assert result["credential_exposed"] is False  # 执行本行的状态、计算或校验逻辑。
assert "opaque-secret-never-shown" not in str(result)  # 执行本行的状态、计算或校验逻辑。

## 7. 日志只保留授权指纹和决策，不记录 bearer secret

审计需要回答谁、代表谁、在哪个策略版本下、为何调用何种工具、参数摘要是什么、票据何时过期以及结果如何；它不需要保存真实 token。日志脱敏还要覆盖异常堆栈、HTTP header、prompt 与工具返回。指纹用于关联事件，不应被误当成可公开的匿名数据。

In [ ]:
audit = {"run_id": action["run_id"], "sub": capability["sub"], "tenant": capability["tenant"], "aud": capability["aud"], "scope": tuple(sorted(capability["scope"])), "digest": digest, "nonce": capability["nonce"], "decision": result["status"]}  # 执行本行的状态、计算或校验逻辑。
assert audit["decision"] == "accepted"  # 执行本行的状态、计算或校验逻辑。
assert "token" not in audit and "secret" not in audit  # 执行本行的状态、计算或校验逻辑。
assert audit["digest"] == capability["action_digest"]  # 执行本行的状态、计算或校验逻辑。

## 8. 撤销、故障与验收必须覆盖整个 Agent Loop

用户取消任务、审批撤销、policy 升级、run 超时或行为异常时，broker 应停止签发并撤销未消费票据；工具调用失败需区分未执行、已执行和未知。验收至少覆盖错误 audience/scope、过期、参数替换、跨租户、重放、日志泄露、撤销竞态和 broker 不可用时 fail-closed。

In [ ]:
tests = {"wrong_audience_blocked": not redeem({**capability, "nonce": "n-2"}, action, "storage.example", "refund:create", 12, set()), "expired_blocked": not redeem({**capability, "nonce": "n-3"}, action, "billing.example", "refund:create", 25, set()), "secret_absent": not result["credential_exposed"]}  # 执行本行的状态、计算或校验逻辑。
assert all(tests.values())  # 执行本行的状态、计算或校验逻辑。
assert tests["wrong_audience_blocked"]  # 执行本行的状态、计算或校验逻辑。
assert tests["expired_blocked"]  # 执行本行的状态、计算或校验逻辑。

## 面试总结

回答这道题可以沿候选动作、确定性授权、参数摘要、一次性 capability、broker 代调用、业务后置条件和脱敏审计展开。核心原则是 Agent 需要的是受限访问能力，而不是密钥本身；认证身份、授权决策和模型推理必须分层。真实系统还需要 HSM/KMS、签名验证、mTLS、工作负载身份、原子 nonce、撤销传播、SIEM 与红队演练。